# Module 2: RDDs (Resilient Distributed Datasets)

RDDs are Spark's **foundational abstraction** — the low-level building block that DataFrames are built on top of.

**Why learn RDDs if DataFrames are better?** Because understanding RDDs teaches you how Spark actually works. When a DataFrame query is slow, you need to understand partitions, shuffles, and the DAG — all RDD concepts.

### Four properties that define an RDD:

- **Resilient**: If a partition is lost (machine crashes), Spark re-computes it from the lineage (the chain of transformations)
- **Distributed**: Data is split across partitions, processed in parallel across the cluster
- **Immutable**: Once created, an RDD cannot be changed. Every transformation creates a NEW RDD
- **Lazy**: Transformations build a plan. Only actions trigger execution

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Module 02 - RDDs") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

---
## Concept 1: Creating RDDs

Two main ways to create an RDD:
- `sc.parallelize(list)` — distribute a Python collection across the cluster
- `sc.textFile(path)` — read a text file, each line becomes an element

When you `parallelize`, Spark splits your data into **partitions**. Each partition is processed independently by a separate task. More partitions = more parallelism.

In [ ]:
# From a Python list
numbers = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

print(f"Type: {type(numbers)}")
print(f"Number of partitions: {numbers.getNumPartitions()}")
print(f"First 3 elements: {numbers.take(3)}")

In [ ]:
# From a text file — each LINE becomes one element
lines = sc.textFile("../data/employees.csv")
print(f"Total lines: {lines.count()}")
print(f"First line: {lines.first()}")

#### Try It: Create Your Own RDDs

1. Create an RDD from a list of 20 numbers (1 to 20) using `sc.parallelize(range(1, 21))`
2. Check how many partitions it has with `.getNumPartitions()`
3. Use `.take(5)` to see the first 5 elements

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
my_rdd = sc.parallelize(range(1, 21))
print(f"Partitions: {my_rdd.getNumPartitions()}")
print(f"First 5: {my_rdd.take(5)}")

---
## Concept 2: Transformations (Lazy)

Transformations create a **new RDD** from an existing one. They are **lazy** — Spark records the instruction but does NOT execute it.

This is a critical concept in Big Data: laziness lets Spark optimize the entire computation chain before running anything.

Common transformations:
- `map(f)` — apply function f to each element
- `filter(f)` — keep elements where f returns True

In [ ]:
# These are TRANSFORMATIONS — nothing executes!
squared = numbers.map(lambda x: x ** 2)
evens = numbers.filter(lambda x: x % 2 == 0)

# Proof: these are still RDD objects, not results
print(f"squared is a: {type(squared)}")
print(f"evens is a: {type(evens)}")
print("Nothing has been computed yet!")

#### Try It: Write Lazy Transformations

Using the `numbers` RDD (1 to 10):
1. Create `cubed` — each number raised to the power of 3 (use `map`)
2. Create `odds` — only odd numbers (use `filter`)
3. Print the **type** of both — they should be `PipelinedRDD`, NOT Python lists

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
cubed = numbers.map(lambda x: x ** 3)
odds = numbers.filter(lambda x: x % 2 != 0)
print(f"cubed type: {type(cubed)}")
print(f"odds type: {type(odds)}")

---
## Concept 3: Actions (Trigger Execution)

Actions **trigger the computation** and return a result to the driver program. This is when Spark actually does work.

Common actions:
- `collect()` — return ALL elements as a Python list (careful: pulls everything to driver memory!)
- `count()` — number of elements
- `take(n)` — return first n elements
- `reduce(f)` — aggregate all elements using function f
- `first()` — return the first element

**Production warning**: Never call `.collect()` on a large RDD — it pulls billions of rows into your driver's memory and crashes it. Use `.take(n)` instead.

In [ ]:
# NOW the computation runs (actions trigger execution)
print(f"Squared: {squared.collect()}")
print(f"Evens: {evens.collect()}")
print(f"Sum: {numbers.reduce(lambda a, b: a + b)}")
print(f"Count: {numbers.count()}")

#### Try It: Trigger Actions

Using the `cubed` and `odds` RDDs you created above:
1. `.collect()` the cubed values
2. `.count()` the odd numbers
3. Use `.reduce(lambda a, b: a + b)` to find the sum of all odd numbers

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
print(f"Cubed: {cubed.collect()}")
print(f"Odd count: {odds.count()}")
print(f"Sum of odds: {odds.reduce(lambda a, b: a + b)}")

---
## Concept 4: flatMap

`flatMap` is one of the most important operations in distributed computing. It's like `map`, but each input can produce **zero or more** outputs, and the results are flattened.

**Real-world use**: Splitting log lines into individual fields, tokenizing text into words, expanding nested data.

In [ ]:
sentences = sc.parallelize([
    "hello world",
    "hello spark",
    "spark is great"
])

# map: each sentence -> a LIST of words (nested — list inside list)
mapped = sentences.map(lambda s: s.split(" "))
print(f"map result: {mapped.collect()}")

# flatMap: each sentence -> individual words (flattened)
flat_mapped = sentences.flatMap(lambda s: s.split(" "))
print(f"flatMap result: {flat_mapped.collect()}")

See the difference? `map` gives you `[["hello", "world"], ["hello", "spark"], ...]` — nested lists. `flatMap` gives you `["hello", "world", "hello", "spark", ...]` — flat.

#### Try It: Use flatMap

1. Create an RDD with 3 sentences of your own
2. Use `flatMap` to split them into individual words
3. Count the total number of words using `.count()`

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
my_text = sc.parallelize([
    "data engineering is fun",
    "spark processes big data",
    "distributed systems scale well"
])
words = my_text.flatMap(lambda s: s.split(" "))
print(f"All words: {words.collect()}")
print(f"Total words: {words.count()}")

---
## Concept 5: Key-Value RDDs and reduceByKey

When RDD elements are `(key, value)` tuples, Spark unlocks powerful operations:
- `reduceByKey(f)` — combine values that share the same key using function f
- `groupByKey()` — group all values for each key (less efficient than reduceByKey)
- `sortByKey()` — sort by key

This is the foundation of **MapReduce** — the programming model that started Big Data. The classic example: **word count**.

In [ ]:
# Classic Word Count — the "Hello World" of Big Data
text = sc.parallelize([
    "the quick brown fox jumps over the lazy dog",
    "the fox was quick and the dog was lazy",
    "spark makes distributed computing easy"
])

word_counts = (
    text
    .flatMap(lambda line: line.split(" "))     # Step 1: Split lines into words
    .map(lambda word: (word, 1))                # Step 2: Each word -> (word, 1)
    .reduceByKey(lambda a, b: a + b)            # Step 3: Sum counts per word
    .sortBy(lambda x: x[1], ascending=False)    # Step 4: Sort by count descending
)

print("Word counts:")
for word, count in word_counts.collect():
    print(f"  {word}: {count}")

**Why `reduceByKey` and not `groupByKey`?** `reduceByKey` combines values locally on each partition BEFORE shuffling data across the network. `groupByKey` sends ALL data across the network first, then combines. For large datasets, `reduceByKey` is dramatically faster.

#### Try It: Count Word Frequencies

Create your own text RDD with at least 3 lines. Use the word count pattern:
`flatMap` -> `map` to `(word, 1)` -> `reduceByKey` -> `sortBy`

Print the top 5 most frequent words.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
my_text = sc.parallelize([
    "spark is a distributed computing engine",
    "spark can process data in parallel",
    "data processing is fast with spark",
    "parallel computing is the future of data"
])

my_counts = (
    my_text
    .flatMap(lambda line: line.split(" "))
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

print("Top 5 words:")
for word, count in my_counts.take(5):
    print(f"  {word}: {count}")

---
## Concept 6: RDD Operations on Real Files

Let's apply RDD operations to our employees CSV. When you read a CSV as text with `sc.textFile()`, each line is a raw string — you need to parse it yourself. This is where DataFrames shine (they handle parsing for you), but doing it with RDDs teaches you what's happening under the hood.

In [ ]:
# Read employees.csv as raw text lines
raw_lines = sc.textFile("../data/employees.csv")

# The first line is the header — we need to skip it
header = raw_lines.first()
print(f"Header: {header}")

# Filter out the header, split each line by comma
employees_rdd = (
    raw_lines
    .filter(lambda line: line != header)        # Skip header
    .map(lambda line: line.split(","))          # Split into fields
)

# Count employees per city (city is index 5)
city_counts = (
    employees_rdd
    .map(lambda fields: (fields[5], 1))         # (city, 1)
    .reduceByKey(lambda a, b: a + b)            # Sum per city
    .sortBy(lambda x: x[1], ascending=False)
)

print("\nEmployees per city:")
for city, count in city_counts.collect():
    print(f"  {city}: {count}")

#### Try It: Analyze Salaries with RDDs

Using the same `employees_rdd` (already split by comma):
1. Create `(city, salary)` pairs — city is index 5, salary is index 3
2. Use `reduceByKey` to find the **total salary** per city
3. Print the results

Hint: You'll need to convert salary to int: `int(fields[3])`

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
total_salary_by_city = (
    employees_rdd
    .map(lambda fields: (fields[5], int(fields[3])))   # (city, salary)
    .reduceByKey(lambda a, b: a + b)                   # Sum salaries per city
    .sortBy(lambda x: x[1], ascending=False)
)

print("Total salary per city:")
for city, total in total_salary_by_city.collect():
    print(f"  {city}: ${total:,}")

---
## When to Use RDDs vs DataFrames

| | RDDs | DataFrames |
|---|---|---|
| **Best for** | Unstructured data, fine-grained control | Structured/tabular data |
| **Optimization** | None (you control everything) | Catalyst optimizer (automatic) |
| **Performance** | Slower (no optimization) | Faster (optimized execution plans) |
| **API** | map, filter, reduce | select, filter, groupBy, SQL |

**Rule of thumb**: Use DataFrames for 95% of work. Use RDDs when you need low-level control over partitioning or are working with non-tabular data.

---
## Capstone Exercise

Build a **character frequency counter**:

1. Create an RDD with at least 3 sentences
2. Use `flatMap` to split into individual **characters** (hint: a string is iterable — `flatMap(lambda s: list(s))`)
3. Filter out spaces
4. Use the `(char, 1)` -> `reduceByKey` pattern to count each character
5. Sort by frequency descending and print the top 10

In [ ]:
# Your capstone code here


In [ ]:
# --- Capstone Solution ---
text = sc.parallelize([
    "apache spark is a distributed computing engine",
    "it processes big data in parallel across clusters",
    "spark supports batch streaming and machine learning"
])

char_counts = (
    text
    .flatMap(lambda s: list(s))          # Each string -> individual characters
    .filter(lambda c: c != " ")          # Remove spaces
    .map(lambda c: (c, 1))               # (char, 1)
    .reduceByKey(lambda a, b: a + b)     # Sum per character
    .sortBy(lambda x: x[1], ascending=False)
)

print("Top 10 characters:")
for char, count in char_counts.take(10):
    print(f"  '{char}': {count}")

In [ ]:
spark.stop()

---
## What You Learned

- **RDDs** are immutable, distributed collections — Spark's low-level API
- **Transformations** (`map`, `filter`, `flatMap`) are lazy — they build a plan
- **Actions** (`collect`, `count`, `reduce`, `take`) trigger execution
- **flatMap** flattens nested results — essential for tokenization
- **reduceByKey** aggregates by key efficiently (better than `groupByKey`)
- The **word count pattern** (`flatMap` -> `map(x, 1)` -> `reduceByKey`) is the foundation of MapReduce

Next: **Module 3 — DataFrames**, where you'll see how Spark makes all of this easier with schemas and optimization.